In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# !cd ..

df = pd.read_csv("container_ml_ready_weather.csv")

# Convert booleans if needed
for col in df.columns:
    if df[col].dtype == object:
        if set(df[col].dropna().unique()).issubset({"True", "False"}):
            df[col] = df[col].map({"True":1, "False":0})

OUT_DIR = "/Users/kellyg/eurogate-twin-1/weather_viz"
import os
os.makedirs(OUT_DIR, exist_ok=True)

In [19]:
# TEMPERATURE VS DWELL TIME

plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df,
    x="temperature_2m",
    y="dwell_hours",
    alpha=0.3
)

plt.title("Temperature vs Dwell Time")
plt.savefig(f"{OUT_DIR}/temp_vs_dwell.png")
plt.close()

In [21]:
# RAIN VS DWELL

plt.figure(figsize=(6,5))

sns.boxplot(
    data=df,
    x="is_raining",
    y="dwell_hours"
)

plt.title("Rain vs Dwell Time")
plt.savefig(f"{OUT_DIR}/rain_vs_dwell.png")
plt.close()

In [23]:
# PRECIPITATION VS DWELL

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="precipitation",
    y="dwell_hours",
    alpha=0.3
)

plt.title("Precipitation vs Dwell Time")
plt.savefig(f"{OUT_DIR}/precip_vs_dwell.png")
plt.close()

In [24]:
# WIND VS DWELL

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="wind_gusts_10m",
    y="dwell_hours",
    alpha=0.3
)

plt.title("Wind Gusts vs Dwell Time")
plt.savefig(f"{OUT_DIR}/windgust_vs_dwell.png")
plt.close()

In [25]:
# CLOUD VS DWELL

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="cloud_cover",
    y="dwell_hours",
    alpha=0.3
)

plt.title("Cloud Cover vs Dwell Time")
plt.savefig(f"{OUT_DIR}/cloudcover_vs_dwell.png")
plt.close()

In [26]:
# BAD WEATHER VS DWELL

plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="bad_weather_score",
    y="dwell_hours"
)

plt.title("Bad Weather Score vs Dwell Time")
plt.savefig(f"{OUT_DIR}/bad_weather_score_vs_dwell.png")
plt.close()

In [27]:
# WEATHER CORRELATION
weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "bad_weather_score",
    "dwell_hours"
]

corr = df[weather_cols].corr()

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Weather Correlation Matrix")
plt.savefig(f"{OUT_DIR}/weather_correlation_heatmap.png")
plt.close()

In [2]:
import pandas as pd
import requests

INPUT_CSV = "eurogate_container_history.csv"
OUTPUT_CSV = "eurogate_container_history_weather.csv"

LAT = 53.541
LON = 9.958
TIMEZONE = "Europe/Berlin"

HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m"
]

df = pd.read_csv(INPUT_CSV)

df["weather_match_time"] = pd.to_datetime(df["arrivalTime"], errors="coerce", utc=True)
df["weather_hour"] = (
    df["weather_match_time"]
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)
    .dt.round("h")
)

start_date = df["weather_hour"].min().date().isoformat()
end_date = df["weather_hour"].max().date().isoformat()

print(f"Fetching Eurogate weather from {start_date} to {end_date}")

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ",".join(HOURLY_VARS),
    "timezone": TIMEZONE,
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

weather = pd.DataFrame(data["hourly"])
weather["weather_time"] = pd.to_datetime(weather["time"])
weather = weather.drop(columns=["time"])

df_weather = df.merge(
    weather,
    left_on="weather_hour",
    right_on="weather_time",
    how="left"
)

df_weather["is_raining"] = (
    df_weather["precipitation"].fillna(0) > 0
).astype(int)

df_weather["bad_weather_score"] = (
    df_weather["is_raining"]
    + (df_weather["wind_gusts_10m"].fillna(0) > 30).astype(int)
    + (df_weather["cloud_cover"].fillna(0) > 80).astype(int)
)

df_weather.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

print("\nWeather missing fraction:")
weather_cols = HOURLY_VARS + ["is_raining", "bad_weather_score"]
print(df_weather[weather_cols].isna().mean().sort_values(ascending=False))

df_weather.head()

Fetching Eurogate weather from 2024-06-23 to 2025-05-06
Saved: eurogate_container_history_weather.csv

Weather missing fraction:
temperature_2m          0.0
relative_humidity_2m    0.0
precipitation           0.0
rain                    0.0
weather_code            0.0
cloud_cover             0.0
wind_speed_10m          0.0
wind_gusts_10m          0.0
is_raining              0.0
bad_weather_score       0.0
dtype: float64


,containerId,first_seen,last_seen,n_snapshots,arrivalTime,departureTime,arrivalVoyageEta,sizetypeIsoCode,typeCode,reefer,...,relative_humidity_2m,precipitation,rain,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m,weather_time,is_raining,bad_weather_score
0,AAMU2504013,2025-04-13,2025-04-21,9,2025-04-12 12:43:17+00:00,2025-04-14 15:40:06+00:00,2025-04-10 15:00:00+00:00,22K2,TK,False,...,36,0.0,0.0,0,0,2.6,16.9,2025-04-12 15:00:00,0,0
1,AAMU2507060,2025-04-17,2025-04-24,8,2025-04-16 12:00:43+00:00,2025-04-17 13:56:28+00:00,2025-04-16 02:00:00+00:00,22K2,TK,False,...,44,0.0,0.0,1,26,4.6,20.2,2025-04-16 14:00:00,0,0
2,AAMU2507115,2025-04-17,2025-04-24,8,2025-04-16 13:39:03+00:00,2025-04-17 07:35:55+00:00,2025-04-16 02:00:00+00:00,22T0,TK,False,...,46,0.0,0.0,3,84,8.3,24.1,2025-04-16 16:00:00,0,1
3,AAMU2609570,2025-04-01,2025-04-04,4,2025-03-24 07:05:50+00:00,2025-03-28 12:18:27+00:00,2025-03-23 09:00:00+00:00,22T1,TK,False,...,97,0.0,0.0,1,46,2.7,6.1,2025-03-24 08:00:00,0,0
4,AAMU6001439,2025-05-06,2025-05-06,1,2025-05-05 06:35:36+00:00,NaN,2025-05-04 21:00:00+00:00,22T3,TK,False,...,65,0.0,0.0,0,0,8.4,24.1,2025-05-05 09:00:00,0,0
